<a href="https://colab.research.google.com/github/saileepanchbhai/Advance-Machine-Learning-Lab/blob/main/RL3_Setting_Up_Optimal_Action_(Extracting_Policy).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install gymnasium

In [2]:
# Used for numerical operations and to create the Q-table (matrix of state-action values)
import numpy as np
#Import Gymnasium library
# Used to create and interact with Reinforcement Learning environments
import gymnasium as gym
# Import random module
# Used for implementing epsilon-greedy action selection (exploration)
import random

In [8]:
# Create the Frozen Lake environment
# "FrozenLake-v1" is a 4x4 grid world environment
# is_slippery=True makes the surface slippery (stochastic movement)
# This means the agent may not always move in the intended direction
env = gym.make("FrozenLake-v1", is_slippery=True)

# Get the total number of states in the environment
# For 4x4 Frozen Lake → 16 states (0 to 15)
state_space = env.observation_space.n
print("State space:", state_space)

# Get the total number of possible actions
# Frozen Lake has 4 actions:
# 0 = Left, 1 = Down, 2 = Right, 3 = Up
action_space = env.action_space.n
print("Action space:",action_space)

State space: 16
Action space: 4


In [9]:
# Initialize the Q-table with zeros
# Rows represent states (0 to 15 in 4x4 Frozen Lake)
# Columns represent actions (0=Left, 1=Down, 2=Right, 3=Up)
# Initially, all state-action values are set to 0
# The agent will update these values during training
Q = np.zeros((state_space, action_space))
Q

array([[0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.]])

In [10]:
import pandas as pd

states = [f"s{i}" for i in range(state_space)]
actions = [f"A{j}" for j in range(action_space)]
Q = pd.DataFrame(np.zeros((state_space, action_space)),
                 index=states,
                 columns=actions)
print(Q)

      A0   A1   A2   A3
s0   0.0  0.0  0.0  0.0
s1   0.0  0.0  0.0  0.0
s2   0.0  0.0  0.0  0.0
s3   0.0  0.0  0.0  0.0
s4   0.0  0.0  0.0  0.0
s5   0.0  0.0  0.0  0.0
s6   0.0  0.0  0.0  0.0
s7   0.0  0.0  0.0  0.0
s8   0.0  0.0  0.0  0.0
s9   0.0  0.0  0.0  0.0
s10  0.0  0.0  0.0  0.0
s11  0.0  0.0  0.0  0.0
s12  0.0  0.0  0.0  0.0
s13  0.0  0.0  0.0  0.0
s14  0.0  0.0  0.0  0.0
s15  0.0  0.0  0.0  0.0


In [11]:
states = [f"s{i}" for i in range(state_space)]
actions = ["LEFT", "DOWN", "RIGHT", "UP"]
Q = pd.DataFrame(np.zeros((state_space, action_space)),
                 index=states,
                 columns=actions)
print(Q)

     LEFT  DOWN  RIGHT   UP
s0    0.0   0.0    0.0  0.0
s1    0.0   0.0    0.0  0.0
s2    0.0   0.0    0.0  0.0
s3    0.0   0.0    0.0  0.0
s4    0.0   0.0    0.0  0.0
s5    0.0   0.0    0.0  0.0
s6    0.0   0.0    0.0  0.0
s7    0.0   0.0    0.0  0.0
s8    0.0   0.0    0.0  0.0
s9    0.0   0.0    0.0  0.0
s10   0.0   0.0    0.0  0.0
s11   0.0   0.0    0.0  0.0
s12   0.0   0.0    0.0  0.0
s13   0.0   0.0    0.0  0.0
s14   0.0   0.0    0.0  0.0
s15   0.0   0.0    0.0  0.0


In [12]:
# -----------------------------
# Hyperparameters for Q-Learning
# -----------------------------

# Learning Rate (α)
# Determines how much newly learned information overrides old information
# Value range: 0 to 1
# Higher value → Faster learning but may be unstable
alpha = 0.8

# Discount Factor (γ)
# Determines importance of future rewards
# Value close to 1 → Agent values future rewards strongly
gamma = 0.95

# Exploration Rate (ε)
# Probability of choosing a random action (exploration)
# Starts at 1.0 → 100% exploration at beginning
epsilon = 1.0

# Epsilon Decay Rate
# After each episode, epsilon is reduced gradually
# Helps shift from exploration to exploitation
epsilon_decay = 0.995

# Minimum Epsilon
# Ensures agent never completely stops exploring
min_epsilon = 0.01

# Number of training episodes
# Total times the agent will interact with environment
episodes = 5000

# Maximum steps per episode
# Prevents infinite loops if agent never reaches goal
max_steps = 100

In [14]:
actions = ["LEFT", "DOWN", "RIGHT", "UP"]
rewards_per_episode = []

for episode in range(episodes):

    state = env.reset()
    if isinstance(state, tuple):
        state = state[0]

    done = False
    total_reward = 0

    for step in range(max_steps):

        # Convert state index to label
        state_label = f"s{state}"

        # Epsilon-Greedy
        if random.uniform(0,1) < epsilon:
            action = env.action_space.sample()
        else:
            action = np.argmax(Q.loc[state_label].values)

        action_name = actions[action]

        # Take action
        step_result = env.step(action)

        if len(step_result) == 5:
            next_state, reward, done, truncated, info = step_result
            done = done or truncated
        else:
            next_state, reward, done, info = step_result

        next_state_label = f"s{next_state}"

        # Q-learning update
        best_next = np.max(Q.loc[next_state_label].values)

        Q.loc[state_label, action_name] += alpha * (
            reward + gamma * best_next - Q.loc[state_label, action_name]
        )

        state = next_state
        total_reward += reward

        if done:
            break

    rewards_per_episode.append(total_reward)

    # Decay epsilon
    epsilon = max(min_epsilon, epsilon * epsilon_decay)

    # Print progress
    if (episode + 1) % 1000 == 0:
        avg = sum(rewards_per_episode[-1000:]) / 1000
        print(f"Episode {episode+1}, Avg Reward: {avg:.3f}")

# -----------------------------
# Results
# -----------------------------
print("\nTraining Completed\n")

print("Q-table:")
print(Q)

print("\nOptimal Policy:")
policy = Q.idxmax(axis=1)
print(policy)

Episode 1000, Avg Reward: 0.213
Episode 2000, Avg Reward: 0.523
Episode 3000, Avg Reward: 0.521
Episode 4000, Avg Reward: 0.525
Episode 5000, Avg Reward: 0.455

Training Completed

Q-table:
         LEFT          DOWN     RIGHT        UP
s0   0.052769  4.815528e-02  0.001043  0.000942
s1   0.000670  4.570373e-04  0.000735  0.051957
s2   0.027575  3.269641e-04  0.000544  0.000535
s3   0.000250  2.045853e-04  0.000367  0.000523
s4   0.070131  2.410903e-02  0.000291  0.000300
s5   0.000000  0.000000e+00  0.000000  0.000000
s6   0.000003  2.964308e-07  0.017467  0.000017
s7   0.000000  0.000000e+00  0.000000  0.000000
s8   0.000346  1.506595e-02  0.000816  0.072687
s9   0.000878  2.858531e-02  0.000533  0.044934
s10  0.255278  3.690739e-04  0.000352  0.004853
s11  0.000000  0.000000e+00  0.000000  0.000000
s12  0.000000  0.000000e+00  0.000000  0.000000
s13  0.131017  2.907754e-02  0.220078  0.136079
s14  0.298329  8.662388e-01  0.267941  0.311172
s15  0.000000  0.000000e+00  0.000000  0.0

In [18]:

# -----------------------------
# Optimal Action for One State
# -----------------------------

actions = ["LEFT", "DOWN", "RIGHT", "UP"]

# Choose a state
state = "s11"

# Get best action index
best_action_index = np.argmax(Q.loc[state].values)

# Convert to action name
best_action = actions[best_action_index]

print(f"Optimal action for {state} --> {best_action}")

Optimal action for s11 --> LEFT
